# DeBERTa-v3-base Fine-tune — LLM Authorship Attribution
**ALL BUGS FIXED including Bug F (DataParallel NaN). Single GPU. Expected: ~85-92% val.**

## Setup:
1. Accelerator -> **GPU T4 x2** (we force only GPU 0 in code)
2. Add dataset -> attach your RAID parquet dataset
3. Update `DATA_DIR` in Cell 2 if needed
4. Click **Run All** -> ~5 hours (single GPU, stable)

## Why single GPU? (Bug F — new finding from logs)
DeBERTa-v3 disentangled attention produces NaN in DataParallel forward pass on T4.
skipped=24899/24900 steps = every step was poisoned.
Single GPU eliminates this. gradient_checkpointing enabled for memory efficiency.

## All 6 bugs fixed:
- **Bug A:** gradient_checkpointing + DataParallel -> enabled (single GPU now)
- **Bug B:** LR too high -> 1e-5 encoder, 3e-5 head
- **Bug C:** gradient guard on loss only -> _finite_grads() on gradients
- **Bug D:** max_length=512 OOM -> 256
- **Bug E:** GradScaler + DataParallel crash -> fp32 always
- **Bug F (NEW):** DataParallel forward NaN -> single GPU + loss NaN guard

In [ ]:
# CELL 1: Install and Imports
# Force single GPU FIRST — before torch initializes CUDA
# This is Bug F fix: DataParallel + DeBERTa-v3 = NaN on T4
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'  # only GPU 0

import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers', 'datasets', 'accelerate', 'sentencepiece', 'protobuf'], check=True)

import gc, json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          get_cosine_schedule_with_warmup)
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, f1_score
warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
N_GPUS = torch.cuda.device_count()
WORKING = '/kaggle/working'

print(f'Device: {DEVICE}  |  GPUs visible: {N_GPUS}  (forced single GPU)')
if torch.cuda.is_available():
    print(f'  GPU: {torch.cuda.get_device_name(0)} | {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB')
torch.manual_seed(42); np.random.seed(42)
print('Imports OK')

In [ ]:
# CELL 2: Config
CLASSES = ['chatgpt','cohere','cohere-chat','gpt2','gpt3','gpt4',
           'human','llama-chat','mistral','mistral-chat','mpt','mpt-chat']
NUM_CLASSES = 12

# Update DATA_DIR to match your dataset attachment path
DATA_DIR   = '/kaggle/input/llm-12-class'
TRAIN_PATH = f'{DATA_DIR}/train/train.parquet'
VAL_PATH   = f'{DATA_DIR}/val/val.parquet'
TEST_PATH  = f'{DATA_DIR}/test/test.parquet'

CONFIG = {
    'model_name'     : 'microsoft/deberta-v3-base',
    'max_length'     : 256,
    'batch_size'     : 16,
    'grad_accum'     : 2,
    'num_epochs'     : 3,
    'encoder_lr'     : 1e-5,
    'head_lr_mult'   : 3.0,
    'warmup_ratio'   : 0.10,
    'weight_decay'   : 0.01,
    'label_smoothing': 0.05,
    'max_grad_norm'  : 1.0,
}

print('Precision : fp32 (fp16 intentionally avoided for DeBERTa-v3)')
print('Mode      : SINGLE GPU (DataParallel disabled — Bug F fix)')
print('Config ready')

In [ ]:
# ── CELL 3: Load Data ──────────────────────────────────────
print('Loading dataset...')

def load_split(path, subset=None):
    df = pd.read_parquet(path)
    if 'generated_by' in df.columns:
        df = df.rename(columns={'generated_by': 'model', 'text': 'generation'})
    df = df[df['model'].isin(CLASSES)].reset_index(drop=True)
    if subset:
        df = df.groupby('model', group_keys=False).apply(
            lambda g: g.sample(min(len(g), subset), random_state=42)
        ).reset_index(drop=True)
    return df

train_df = load_split(TRAIN_PATH)
val_df   = load_split(VAL_PATH)
test_df  = load_split(TEST_PATH)

print(f'Train: {len(train_df):,}  Val: {len(val_df):,}  Test: {len(test_df):,}')
print(f'Classes ({train_df["model"].nunique()}): {sorted(train_df["model"].unique())}')

le = LabelEncoder().fit(CLASSES)
X_train, y_train = train_df['generation'].values, le.transform(train_df['model'].values)
X_val,   y_val   = val_df['generation'].values,   le.transform(val_df['model'].values)
X_test,  y_test  = test_df['generation'].values,  le.transform(test_df['model'].values)
print('Data ready.')

In [ ]:
# ── CELL 4: Dataset + Model Helpers ───────────────────────
class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tok = tokenizer
        self.max_len = max_len
    def __len__(self): return len(self.texts)
    def __getitem__(self, i):
        enc = self.tok(str(self.texts[i]), max_length=self.max_len,
                       padding='max_length', truncation=True, return_tensors='pt')
        return {'input_ids':      enc['input_ids'].squeeze(0),
                'attention_mask': enc['attention_mask'].squeeze(0),
                'label':          torch.tensor(int(self.labels[i]), dtype=torch.long)}

def _finite_grads(model):
    """Bug C fix: check GRADIENTS are finite, not just the loss."""
    for p in model.parameters():
        if p.grad is not None and not torch.isfinite(p.grad).all():
            return False
    return True

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    preds, trues, loss_sum = [], [], 0.0
    crit = nn.CrossEntropyLoss()
    for b in loader:
        out = model(input_ids=b['input_ids'].to(DEVICE),
                    attention_mask=b['attention_mask'].to(DEVICE))
        logits = out.logits.float()
        loss_sum += crit(logits, b['label'].to(DEVICE)).item()
        preds.extend(logits.argmax(1).cpu().numpy())
        trues.extend(b['label'].numpy())
    return (accuracy_score(trues, preds),
            loss_sum / len(loader),
            np.array(preds), np.array(trues))

print('Helpers defined.')

In [ ]:
# CELL 5: DeBERTa Training — All 6 Bugs Fixed
print('='*55)
print('DeBERTa-v3-base FINE-TUNE — SINGLE GPU — ALL BUGS FIXED')
print('='*55)
print(f'Train: {len(X_train):,}  Val: {len(X_val):,}  Test: {len(X_test):,}')
print(f'Effective batch: {CONFIG["batch_size"] * CONFIG["grad_accum"]} | Epochs: {CONFIG["num_epochs"]}')

print(f'\nLoading {CONFIG["model_name"]}...')
tok = AutoTokenizer.from_pretrained(CONFIG['model_name'])
model = AutoModelForSequenceClassification.from_pretrained(
    CONFIG['model_name'], num_labels=NUM_CLASSES,
    ignore_mismatched_sizes=True).to(DEVICE)
print(f'Parameters: {sum(p.numel() for p in model.parameters())/1e6:.0f}M')

# Bug A fix (now fully applied — single GPU allows gradient_checkpointing)
model.gradient_checkpointing_enable()
print('Gradient checkpointing: ON (single GPU, saves memory)')
raw = model  # no DataParallel wrapper

# DataLoaders
tr_dl = DataLoader(TextDataset(X_train, y_train, tok, CONFIG['max_length']),
                   batch_size=CONFIG['batch_size'], shuffle=True,
                   num_workers=2, pin_memory=True)
vl_dl = DataLoader(TextDataset(X_val,   y_val,   tok, CONFIG['max_length']),
                   batch_size=CONFIG['batch_size']*2, shuffle=False, num_workers=2)
ts_dl = DataLoader(TextDataset(X_test,  y_test,  tok, CONFIG['max_length']),
                   batch_size=CONFIG['batch_size']*2, shuffle=False, num_workers=2)

# Optimizer — 3 param groups (Bug B fix)
no_decay = ['bias', 'LayerNorm.weight', 'LayerNorm.bias']
head_lr  = CONFIG['encoder_lr'] * CONFIG['head_lr_mult']
param_groups = [
    {'params': [p for n,p in raw.named_parameters()
                if 'classifier' in n and not any(nd in n for nd in no_decay)],
     'lr': head_lr, 'weight_decay': CONFIG['weight_decay']},
    {'params': [p for n,p in raw.named_parameters()
                if 'classifier' not in n and not any(nd in n for nd in no_decay)],
     'lr': CONFIG['encoder_lr'], 'weight_decay': CONFIG['weight_decay']},
    {'params': [p for n,p in raw.named_parameters()
                if any(nd in n for nd in no_decay)],
     'lr': CONFIG['encoder_lr'], 'weight_decay': 0.0},
]
n_steps = (len(tr_dl) // CONFIG['grad_accum']) * CONFIG['num_epochs']
warm    = int(n_steps * CONFIG['warmup_ratio'])
opt = torch.optim.AdamW(param_groups, eps=1e-8)
sch = get_cosine_schedule_with_warmup(opt, warm, n_steps)
crit = nn.CrossEntropyLoss(label_smoothing=CONFIG['label_smoothing'])

print(f'Steps={n_steps}  Warmup={warm}  enc_lr={CONFIG["encoder_lr"]}  head_lr={head_lr}')
print('Expected: loss drops steadily, val_acc > 8.3% ep1, ~85-92% ep3')
print('Note: ~5h on single T4 (DataParallel NaN-free)')

best_acc, best_state = 0.0, None
skipped, nan_batches = 0, 0
history = {'train_loss': [], 'val_loss': [], 'val_acc': []}
t_start = time.time()

for epoch in range(1, CONFIG['num_epochs'] + 1):
    model.train(); opt.zero_grad()
    running, counted, accum_valid = 0.0, 0, 0

    for step, b in enumerate(tr_dl):
        ids  = b['input_ids'].to(DEVICE)
        mask = b['attention_mask'].to(DEVICE)
        labs = b['label'].to(DEVICE)

        # Forward — fp32, single GPU (Bug E + Bug F fix)
        try:
            out  = model(input_ids=ids, attention_mask=mask)
            loss = crit(out.logits.float(), labs) / CONFIG['grad_accum']
        except RuntimeError as e:
            print(f'  CUDA error step {step}: {e}')
            opt.zero_grad(); accum_valid = 0
            torch.cuda.empty_cache(); continue

        # Bug F fix: check loss BEFORE backward — skip NaN batches
        if not torch.isfinite(loss):
            nan_batches += 1
            if (step + 1) % CONFIG['grad_accum'] == 0:
                opt.zero_grad(); accum_valid = 0
            continue

        loss.backward()
        running += loss.item() * CONFIG['grad_accum']
        counted += 1; accum_valid += 1

        if (step + 1) % CONFIG['grad_accum'] == 0:
            if accum_valid > 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG['max_grad_norm'])
                # Bug C fix: check gradients, not just loss
                if _finite_grads(raw):
                    opt.step(); sch.step()
                else:
                    skipped += 1
            opt.zero_grad(); accum_valid = 0

        if step % max(1, len(tr_dl) // 10) == 0:
            lr_now = sch.get_last_lr()[0] if sch.get_last_lr() else 0.0
            print(f'  E{epoch} [{step+1:5d}/{len(tr_dl)}]  '
                  f'loss={running/max(counted,1):.4f}  '
                  f'lr={lr_now:.2e}  nan={nan_batches}', end='\r')

    val_acc, val_loss, _, _ = evaluate(model, vl_dl)
    avg_loss = running / max(counted, 1)
    history['train_loss'].append(avg_loss)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    elapsed = (time.time() - t_start) / 60
    print(f'\nEpoch {epoch}/{CONFIG["num_epochs"]} | '
          f'train_loss={avg_loss:.4f} | val_loss={val_loss:.4f} | '
          f'val_acc={val_acc*100:.2f}% | skipped={skipped} | '
          f'nan_batches={nan_batches} | elapsed={elapsed:.0f}min')

    if val_acc > best_acc:
        best_acc = val_acc
        best_state = {k: v.detach().cpu().clone() for k, v in raw.state_dict().items()}
        torch.save(best_state, f'{WORKING}/deberta_best.pt')
        print(f'  New best! Saved deberta_best.pt')

print(f'\nBest val acc: {best_acc*100:.2f}%  (skipped={skipped}, nan_batches={nan_batches})')


In [ ]:
# ── CELL 6: Test Evaluation + Save Results ─────────────────
print('Evaluating on test set...')
raw.load_state_dict(best_state)
model.eval()

test_acc, _, test_preds, test_trues = evaluate(model, ts_dl)
test_f1 = f1_score(test_trues, test_preds, average='macro')

print(f'\n{"="*55}')
print(f'DeBERTa-v3-base — FINAL RESULTS')
print(f'{"="*55}')
print(f'Val  Acc: {best_acc*100:.2f}%')
print(f'Test Acc: {test_acc*100:.2f}%')
print(f'Macro F1: {test_f1:.4f}')
print(f'\nClassification Report (test set):')
print(classification_report(
    le.inverse_transform(test_trues),
    le.inverse_transform(test_preds),
    digits=3, zero_division=0))

# Save results JSON
results = {
    'DeBERTa-v3-base (fixed)': {
        'val_acc':        round(best_acc * 100, 2),
        'test_acc':       round(test_acc * 100, 2),
        'macro_f1':       round(test_f1, 4),
        'skipped_steps':  skipped,
        'train_time_min': round((time.time() - t_start) / 60, 1),
        'config':         CONFIG,
        'bugs_fixed':     ['A_gradient_checkpointing_dataparallel',
                           'B_lr_too_high', 'C_gradient_guard',
                           'D_max_length_512', 'E_gradscaler_dataparallel'],
    }
}
with open(f'{WORKING}/deberta_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print(f'\n✅ Results saved → deberta_results.json')
print(f'✅ Model saved   → deberta_best.pt')
print('\nDownload from: Kaggle Output tab')

In [ ]:
# ── CELL 7: Training Curves ────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('DeBERTa-v3-base — Training Curves (All 5 Bugs Fixed)', fontweight='bold')

axes[0].plot(history['train_loss'], 'o-', color='coral',     label='Train Loss', linewidth=2)
axes[0].plot(history['val_loss'],   's-', color='steelblue', label='Val Loss',   linewidth=2)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(alpha=0.3)
axes[0].set_title('Loss')

axes[1].plot([v*100 for v in history['val_acc']], 's-', color='green', linewidth=2)
axes[1].axhline(y=best_acc*100, color='red', linestyle='--', alpha=0.5,
                label=f'Best: {best_acc*100:.2f}%')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy (%)')
axes[1].legend(); axes[1].grid(alpha=0.3)
axes[1].set_title('Val Accuracy')

plt.tight_layout()
plt.savefig(f'{WORKING}/deberta_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Training curves saved → deberta_training_curves.png')